In [1]:
import sys
# the mock-0.3.1 dir contains testcase.py, testutils.py & mock.py
sys.path.append('/data/user/wu_j1/bamf')

In [5]:
import numpy as np
import pandas as pd

from analysis_codes.itx_helper import *

folder = "C:/Users\wu_j1-adm\switchdrive/"

folder = "D:/work/Data/"
folder = "/data/user/wu_j1/"

filename = "Zurich-ToF-ACSM/raw/input_matrices_martavia_noised_20220315.itx"
filename = "Krakow/raw/KRK_PMF_input_new.itx"
filename="Krakow/raw/KRK_PMF_ts.itx"
mz2remove = [16,17,18,19,20,28,12,37]

# read 

In [6]:
waves = itx_to_pandas(folder + filename)
waves.keys()


dict_keys(['WAVES/D'])

In [9]:
waves['WAVES/D']
df = waves['WAVES/D']
df

,DateUTC,HOA,BBOA,CCOA,MO_OOA,LO_OOA,OOA
0,3.598270e+09,1.088912,0.502180,1.128500,1.522545,0.383934,1.906480
1,3.598272e+09,1.037656,0.020797,1.303923,0.680600,0.384180,1.064781
2,3.598274e+09,0.849802,0.670091,1.091423,0.938110,0.836266,1.774376
3,3.598276e+09,1.065534,0.239267,0.944973,1.499861,0.695237,2.195098
4,3.598277e+09,1.006902,0.188146,1.134418,2.003491,0.947114,2.950605
...,...,...,...,...,...,...,...
16268,3.637717e+09,0.878920,0.825180,0.727054,5.379306,4.467822,9.847128
16269,3.637719e+09,0.744441,0.425404,0.735912,5.869260,4.158776,10.028036
16270,3.637721e+09,0.978829,0.567442,0.767273,5.243870,3.730336,8.974206
16271,3.637723e+09,0.830700,0.394445,0.807225,5.543607,4.784796,10.328404


In [76]:
def igor_time_convert(t):
    #     t_ls = np.array(t_ls).reshape(-1)
    return np.timedelta64(int(t), "s") + np.datetime64("1904-01-01")

df["DateUTC"] = df["DateUTC"].apply(igor_time_convert)
# igor_time_convert

In [77]:
df

,DateUTC,HOA,BBOA,CCOA,MO_OOA,LO_OOA,OOA'
0,2018-01-08 15:30:00,1.088912,0.502180,1.128500,1.522545,0.383934,1.906480
1,2018-01-08 16:00:00,1.037656,0.020797,1.303923,0.680600,0.384180,1.064781
2,2018-01-08 16:30:00,0.849802,0.670091,1.091423,0.938110,0.836266,1.774376
3,2018-01-08 17:00:00,1.065534,0.239267,0.944973,1.499861,0.695237,2.195098
4,2018-01-08 17:30:00,1.006902,0.188146,1.134418,2.003491,0.947114,2.950605
...,...,...,...,...,...,...,...
16268,2019-04-10 05:00:00,0.878920,0.825180,0.727054,5.379306,4.467822,9.847128
16269,2019-04-10 05:30:00,0.744441,0.425404,0.735912,5.869260,4.158776,10.028036
16270,2019-04-10 06:00:00,0.978829,0.567442,0.767273,5.243870,3.730336,8.974206
16271,2019-04-10 06:30:00,0.830700,0.394445,0.807225,5.543607,4.784796,10.328404


In [80]:
from scipy.stats import (binom_test, kruskal, ks_2samp, pearsonr, spearmanr)

coeff, _ = spearmanr(df["HOA"],df["BBOA"],axis=None)

In [82]:
coeff

0.7975798138477674

# normal

In [54]:
waves = itx_to_pandas(folder + filename)
waves.keys()

dict_keys(['amus', 'Org_Specs_30', 'OrgSpecs_err_30', 'acsm_utc_time_30'])

In [23]:
Specs = waves['Org_Specs_30']
Specs.shape

(16286, 93)

In [24]:
error = waves['OrgSpecs_err_30']
error.shape

(16286, 93)

In [25]:
amus_nb = waves['amus']
amus_nb.shape

(93, 1)

In [55]:
timelist = waves['acsm_utc_time_30']
timelist.shape

(16286, 1)

In [28]:
Specs = remove_mz(mz2remove,Specs,amus_nb)
error = remove_mz(mz2remove,error,amus_nb)

In [65]:
def igor_time_convert(t):
    #     t_ls = np.array(t_ls).reshape(-1)
    return np.timedelta64(int(t), "s") + np.datetime64("1904-01-01")
time = timelist.apply(igor_time_convert,axis=1)
time

0       2018-01-08 15:30:00
1       2018-01-08 16:00:00
2       2018-01-08 16:30:00
3       2018-01-08 17:00:00
4       2018-01-08 17:30:00
                ...        
16281   2019-04-10 05:00:00
16282   2019-04-10 05:30:00
16283   2019-04-10 06:00:00
16284   2019-04-10 06:30:00
16285   2019-04-10 07:00:00
Length: 16286, dtype: datetime64[ns]

In [66]:
def igor_time_convert(t_ls):
    t_ls = np.array(t_ls).reshape(-1)
    # return pd.to_datetime(([int(i) for i in t_ls]), unit="s") - pd.Timedelta(
    #     24107, unit="d"
    # )
    return np.vectorize(lambda x: np.timedelta64(x, "s"))(
        t_ls.astype(int)
    ) + np.datetime64("1904-01-01")


time = igor_time_convert(timelist)
time

array(['2018-01-08T15:30:00', '2018-01-08T16:00:00',
       '2018-01-08T16:30:00', ..., '2019-04-10T06:00:00',
       '2019-04-10T06:30:00', '2019-04-10T07:00:00'],
      dtype='datetime64[s]')

In [12]:
amus_nb = np.array(([int(i) for i in amus_nb[0] if i not in [16,17,18,19,20,28,12,37]]))

In [13]:
amus_nb.shape,Specs.shape

((88,), (16286, 88))

In [14]:

Specs.columns=[int(i) for i in amus_nb]
Specs.index = time

# Specs.to_hdf(folder+'/Krakow/processed/KRK_PMF_input_new.h5', key='data', mode='w')  

In [15]:

error.columns=[int(i) for i in amus_nb]
error.index = time
# error
# error.to_hdf(folder+'/Krakow/processed/KRK_PMF_input_new.h5', key='error', mode='r+')  

ValueError: Length mismatch: Expected axis has 93 elements, new values have 88 elements

In [ ]:
error

In [16]:


# filepath = "work\Data/Zurich-ToF-ACSM\input_matrices_martavia_noised_20220315.h5"
# error_df = pd.read_hdf(folder+filepath,"error")
# data_df = pd.read_hdf(folder+filepath,"data")

In [235]:
winter_data_df = Specs[(Specs.index <= pd.Timestamp(2018, 10, 24, 0)) &(Specs.index >= pd.Timestamp(2018, 10, 10, 0))]
winter_error_df = error[(error.index <= pd.Timestamp(2018, 10, 24, 0)) &(error.index >= pd.Timestamp(2018, 10, 10, 0))]

winter_data_df = Specs[(Specs.index <= pd.Timestamp(2019, 4, 11 , 0)) &(Specs.index >= pd.Timestamp(2019, 3, 15,0))]
winter_error_df = error[(error.index <= pd.Timestamp(2019, 4, 11 , 0)) &(error.index >= pd.Timestamp(2019, 3, 15,0))]

winter_data_df.to_hdf(folder+'/Krakow/processed/KRK_PMF_input_Mar.h5', key='data', mode='w')  
winter_error_df.to_hdf(folder+'/Krakow/processed/KRK_PMF_input_Mar.h5', key='error', mode='r+')  

In [236]:
winter_data_df

,13,15,24,25,26,27,29,30,31,38,...,111,112,113,114,115,116,117,118,119,120
2019-03-15 00:00:00,0.029703,0.020632,0.009405,0.006598,0.014205,-0.044705,-0.168997,-0.003718,0.026649,-0.001376,...,0.001002,-0.000941,-0.001225,-0.004957,0.006475,-0.002009,0.006380,-0.002252,0.005069,-0.004928
2019-03-15 00:30:00,0.019801,0.144082,-0.005649,0.007583,0.032078,0.294827,0.215368,0.004738,-0.035643,-0.009083,...,0.008966,-0.000012,0.001370,-0.001010,0.006364,-0.003107,-0.001899,0.009859,0.010218,-0.010445
2019-03-15 01:00:00,-0.013866,0.089870,-0.002489,0.010884,-0.000456,0.113202,-0.064711,-0.001424,-0.017016,0.006304,...,0.009444,0.005210,-0.000008,0.006505,0.002494,0.004612,-0.004567,0.010027,0.006044,0.000804
2019-03-15 01:30:00,0.023351,0.065873,-0.008216,0.006668,0.003860,0.054419,0.108963,0.002397,-0.008056,0.009906,...,-0.002563,0.000111,0.009437,0.008099,0.006800,-0.001460,0.000238,0.002229,0.013719,0.004871
2019-03-15 02:00:00,0.003368,-0.093585,0.007137,0.000037,-0.004751,0.129237,0.026776,0.000589,-0.036255,-0.000585,...,0.003953,-0.004283,-0.002995,0.001445,0.008530,-0.002233,0.003569,0.003422,-0.001403,0.003715
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2019-04-10 05:00:00,0.050378,0.293053,0.019358,0.033263,0.118950,0.379278,1.013143,0.022289,0.200964,0.066579,...,0.022693,0.013698,0.013684,0.011438,0.042559,0.023342,0.015661,0.004957,0.032708,0.014308
2019-04-10 05:30:00,0.090977,0.232242,0.003126,0.051296,0.170406,0.410444,0.714955,0.015729,0.161000,0.042398,...,0.034125,0.013260,0.021385,0.006524,0.024382,0.006796,0.014281,0.007714,0.012998,0.024054
2019-04-10 06:00:00,0.038717,0.399933,0.015264,0.049599,0.181920,0.611250,0.890137,0.019583,0.177183,0.061970,...,0.030354,0.011480,0.014770,0.020186,0.032451,0.002959,0.019330,0.016856,0.026634,0.018623
2019-04-10 06:30:00,-0.026272,0.064406,0.012406,0.020887,0.193198,0.575019,0.926329,0.020379,0.161467,0.046711,...,0.027115,0.013475,0.017990,0.008151,0.032363,0.021530,0.012372,0.013851,0.030022,0.017127


In [231]:
winter_data_df.to_numpy().min(),winter_data_df.to_numpy().max()

(-0.57828069, 6.7475219)

In [21]:
import h5py
f = h5py.File( "/data/user/wu_j1/Krakow/processed/KRK_PMF_input_Oct.h5",'r')

keys = list(f.keys())
keys

['data', 'error']

In [237]:
data_df = pd.read_hdf("/data/user/wu_j1/Krakow/processed/KRK_PMF_input_Oct.h5",key='data')
data_df

,13,15,24,25,26,27,29,30,31,38,...,111,112,113,114,115,116,117,118,119,120
2018-10-10 00:00:00,0.175799,0.658565,0.025535,0.151892,0.560820,0.927342,1.653880,0.036385,0.404394,0.146367,...,0.067928,0.034568,0.056256,0.030947,0.140159,0.037294,0.075556,0.077051,0.104640,0.043850
2018-10-10 00:30:00,0.012904,0.839290,0.013619,0.096882,0.455274,1.051240,1.439890,0.031678,0.370030,0.154513,...,0.080703,0.041472,0.057905,0.037816,0.126709,0.040539,0.071865,0.046504,0.092985,0.058771
2018-10-10 01:00:00,0.205750,0.631972,0.027357,0.079346,0.371946,0.669312,1.420448,0.031250,0.370357,0.131635,...,0.055598,0.026375,0.038373,0.025600,0.103678,0.041245,0.049033,0.045389,0.071540,0.035809
2018-10-10 01:30:00,0.112015,0.747650,0.025685,0.108708,0.462605,0.923408,1.464956,0.032229,0.233655,0.144150,...,0.065221,0.040023,0.049562,0.031883,0.116175,0.041808,0.071943,0.040362,0.089015,0.056187
2018-10-10 02:00:00,0.107304,0.644161,-0.009236,0.077569,0.431414,0.812476,1.213048,0.026687,0.345286,0.138435,...,0.058827,0.033650,0.060404,0.044601,0.093174,0.040599,0.073189,0.037672,0.094948,0.051745
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2018-10-23 13:30:00,0.000566,0.213665,0.003757,-0.012441,0.058066,-0.074852,0.179862,0.003957,-0.008136,0.019964,...,0.001762,0.004152,0.009835,-0.003319,0.012347,0.005982,0.005245,0.005525,0.013881,0.004663
2018-10-23 14:00:00,-0.005736,0.051045,0.000673,0.000924,0.035589,-0.096425,0.206071,0.004534,0.022871,0.017693,...,0.004632,0.005962,0.001573,0.005592,0.007692,0.001313,0.002599,0.008793,0.009559,0.009483
2018-10-23 14:30:00,-0.000380,0.091377,-0.007702,0.005398,0.019671,0.110423,-0.002356,-0.000052,-0.015956,0.003315,...,0.012378,0.006543,-0.001662,0.005726,0.006259,0.004750,-0.000059,0.006693,0.008615,0.003542
2018-10-23 15:00:00,0.026330,0.002285,0.005374,0.016503,0.042838,0.028391,0.217760,0.004791,-0.006757,0.030590,...,0.005852,0.005207,0.002338,0.005920,0.008495,0.003107,0.009966,0.003581,0.011623,0.002873
